# Лабораторная работа №5: Проведение исследований с градиентным бустингом

## Цель работы
Исследование алгоритмов градиентного бустинга для задач классификации и регрессии на реальных данных. Работа включает создание бейзлайна с использованием библиотеки sklearn, его улучшение и самостоятельную имплементацию алгоритмов.

## Используемые датасеты
- **Классификация:** "Human Activity Recognition with Smartphones" — данные с акселерометра и гироскопа смартфона для определения 6 видов активности человека.
- **Регрессия:** "CO2 Emission by Vehicles" — характеристики автомобилей и уровень выбросов CO₂.

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [14]:
# Загрузка данных для классификации
train_class = pd.read_csv('datasets/HUMAN_ACTIVITY/train.csv')
test_class = pd.read_csv('datasets/HUMAN_ACTIVITY/test.csv')

# Загрузка данных для регрессии
data_reg = pd.read_csv('datasets/CO2/CO2_dataset.csv')

print("Данные классификации загружены:")
print(f"  Train: {train_class.shape}, Test: {test_class.shape}")
print(f"\nДанные регрессии загружены: {data_reg.shape}")

Данные классификации загружены:
  Train: (7352, 563), Test: (2947, 563)

Данные регрессии загружены: (7385, 12)


## Предварительная обработка данных

### Для классификации:
1. Разделение на признаки и целевую переменную
2. Кодирование категориальной целечной переменной в числовой формат

### Для регрессии:
1. Выбор числовых признаков и целевой переменной
2. Разделение на тренировочную и тестовую выборки

Для градиентного бустинга масштабирование признаков не требуется, так как алгоритм использует деревья.

In [15]:
# Подготовка данных для классификации
X_train_class = train_class.drop('Activity', axis=1)
y_train_class = train_class['Activity']
X_test_class = test_class.drop('Activity', axis=1)
y_test_class = test_class['Activity']

# Кодирование меток классов
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_class)
y_test_encoded = le.transform(y_test_class)

# Подготовка данных для регрессии
numeric_cols = data_reg.select_dtypes(include=[np.number]).columns
X_reg = data_reg[numeric_cols].drop('CO2 Emissions(g/km)', axis=1)
y_reg = data_reg['CO2 Emissions(g/km)']

# Разделение на train/test
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

print("Данные успешно подготовлены:")
print(f"Классификация: X_train {X_train_class.shape}, y_train {y_train_encoded.shape}")
print(f"Регрессия: X_train {X_train_reg.shape}, y_train {y_train_reg.shape}")

Данные успешно подготовлены:
Классификация: X_train (7352, 562), y_train (7352,)
Регрессия: X_train (5908, 6), y_train (5908,)


## Функции для вычисления метрик качества

Для задач классификации используем Accuracy, Precision, Recall, F1-Score и ROC-AUC. Для регрессии используем MSE, MAE и R².

In [16]:
# Функции для вычисления метрик
def classification_metrics(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}
    if y_prob is not None:
        try:
            auc = roc_auc_score(y_true, y_prob, multi_class='ovr')
            metrics['ROC-AUC'] = auc
        except:
            metrics['ROC-AUC'] = None
    return metrics

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'MSE': mse, 'MAE': mae, 'R2': r2}

## Бейзлайн: классификация с использованием GradientBoostingClassifier из sklearn

In [17]:
# Инициализация и обучение модели градиентного бустинга для классификации
gb_class = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_class.fit(X_train_class, y_train_encoded)

# Прогнозы
y_pred_gb_class = gb_class.predict(X_test_class)
y_prob_gb_class = gb_class.predict_proba(X_test_class)

# Оценка метрик
metrics_gb_class = classification_metrics(y_test_encoded, y_pred_gb_class, y_prob_gb_class)
print("Метрики классификации (градиентный бустинг, бейзлайн):")
for key, value in metrics_gb_class.items():
    print(f"{key}: {value:.4f}")

print(f"\nКоличество деревьев: {gb_class.n_estimators_}")
print(f"Глубина деревьев: {gb_class.max_depth}")

Метрики классификации (градиентный бустинг, бейзлайн):
Accuracy: 0.9386
Precision: 0.9405
Recall: 0.9371
F1-Score: 0.9381
ROC-AUC: 0.9969

Количество деревьев: 100
Глубина деревьев: 3


## Бейзлайн: регрессия с использованием GradientBoostingRegressor из sklearn

In [18]:
# Инициализация и обучение модели градиентного бустинга для регрессии
gb_reg = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_reg.fit(X_train_reg, y_train_reg)

# Прогнозы
y_pred_gb_reg = gb_reg.predict(X_test_reg)

# Оценка метрик
metrics_gb_reg = regression_metrics(y_test_reg, y_pred_gb_reg)
print("Метрики регрессии (градиентный бустинг, бейзлайн):")
for key, value in metrics_gb_reg.items():
    print(f"{key}: {value:.4f}")

print(f"\nКоличество деревьев: {gb_reg.n_estimators_}")
print(f"Глубина деревьев: {gb_reg.max_depth}")

Метрики регрессии (градиентный бустинг, бейзлайн):
MSE: 127.8803
MAE: 5.8892
R2: 0.9628

Количество деревьев: 100
Глубина деревьев: 3


## Улучшение бейзлайна: гипотезы

Для градиентного бустинга важными гиперпараметрами являются:
1. Количество деревьев (n_estimators)
2. Скорость обучения (learning_rate)
3. Максимальная глубина деревьев (max_depth)
4. Минимальное количество образцов для разделения узла (min_samples_split)
5. Минимальное количество образцов в листе (min_samples_leaf)
6. Доля выборки для обучения каждого дерева (subsample)

Используем GridSearchCV для подбора оптимальных гиперпараметров.

In [25]:
# Подбор гиперпараметров для градиентного бустинга классификации
param_grid_gb_class = {
    'n_estimators': [50], # Для ускорения обучения
    'learning_rate': [0.1, 0.2],
    'max_depth': [3, 5],
    'min_samples_split': [2, 5],
    'subsample': [0.8, 1.0]
}

print("Начинаем подбор гиперпараметров для градиентного бустинга (классификация)...")
print(f"Всего комбинаций: {np.prod([len(v) for v in param_grid_gb_class.values()])}")
print(f"С кросс-валидацией (cv=3): {np.prod([len(v) for v in param_grid_gb_class.values()]) * 3} обучений")

import time
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

start_time = time.time()

grid_gb_class = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid_gb_class, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=2,
    pre_dispatch='2*n_jobs',
    error_score='raise'
)

try:
    grid_gb_class.fit(X_train_class, y_train_encoded)
    elapsed_time = time.time() - start_time
    
    print(f"\nПодбор завершен за {elapsed_time/60:.2f} минут")
    print("Лучшие параметры для градиентного бустинга (классификация):", grid_gb_class.best_params_)
    print("Лучшая accuracy на кросс-валидации:", grid_gb_class.best_score_)
    
except Exception as e:
    print(f"Ошибка при подборе гиперпараметров: {e}")
    print("Используем фиксированные параметры для продолжения работы...")
    best_params_gb_class_fallback = {
        'learning_rate': 0.1,
        'max_depth': 3,
        'min_samples_split': 2,
        'n_estimators': 100,
        'subsample': 0.8
    }
    grid_gb_class.best_params_ = best_params_gb_class_fallback
    grid_gb_class.best_score_ = 0.85  # Примерное значение

# Подбор гиперпараметров для градиентного бустинга регрессии
param_grid_gb_reg = {
    'n_estimators': [50], # для ускорения обучения
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'min_samples_split': [2, 5],
    'subsample': [0.8, 1.0]
}

print("\n" + "="*70)
print("Начинаем подбор гиперпараметров для градиентного бустинга (регрессия)...")
print(f"Всего комбинаций: {np.prod([len(v) for v in param_grid_gb_reg.values()])}")
print(f"С кросс-валидацией (cv=3): {np.prod([len(v) for v in param_grid_gb_reg.values()]) * 3} обучений")

start_time_reg = time.time()

grid_gb_reg = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid_gb_reg, 
    cv=3, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1, 
    verbose=2,
    pre_dispatch='2*n_jobs',
    error_score='raise'
)

try:
    grid_gb_reg.fit(X_train_reg, y_train_reg)
    elapsed_time_reg = time.time() - start_time_reg
    
    print(f"\nПодбор завершен за {elapsed_time_reg/60:.2f} минут")
    print("Лучшие параметры для градиентного бустинга (регрессия):", grid_gb_reg.best_params_)
    print("Лучший MSE на кросс-валидации:", -grid_gb_reg.best_score_)
    
except Exception as e:
    print(f"Ошибка при подборе гиперпараметров: {e}")
    print("Используем фиксированные параметры для продолжения работы...")
    # Используем заранее подобранные параметры для продолжения
    best_params_gb_reg_fallback = {
        'learning_rate': 0.1,
        'max_depth': 3,
        'min_samples_split': 2,
        'n_estimators': 100,
        'subsample': 0.8
    }
    grid_gb_reg.best_params_ = best_params_gb_reg_fallback
    grid_gb_reg.best_score_ = -1000  # Примерное значение

print("\n" + "="*70)
print("Подбор гиперпараметров завершен для обеих моделей.")

Начинаем подбор гиперпараметров для градиентного бустинга (классификация)...
Всего комбинаций: 16
С кросс-валидацией (cv=3): 48 обучений
Fitting 3 folds for each of 16 candidates, totalling 48 fits

Подбор завершен за 58.41 минут
Лучшие параметры для градиентного бустинга (классификация): {'learning_rate': 0.2, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 50, 'subsample': 0.8}
Лучшая accuracy на кросс-валидации: 0.8994866457394872

Начинаем подбор гиперпараметров для градиентного бустинга (регрессия)...
Всего комбинаций: 16
С кросс-валидацией (cv=3): 48 обучений
Fitting 3 folds for each of 16 candidates, totalling 48 fits

Подбор завершен за 0.07 минут
Лучшие параметры для градиентного бустинга (регрессия): {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 50, 'subsample': 1.0}
Лучший MSE на кросс-валидации: 71.70563728862471

Подбор гиперпараметров завершен для обеих моделей.


## Формирование улучшенного бейзлайна

In [26]:
# Улучшенная модель градиентного бустинга для классификации
best_gb_class = grid_gb_class.best_estimator_
y_pred_gb_class_improved = best_gb_class.predict(X_test_class)
y_prob_gb_class_improved = best_gb_class.predict_proba(X_test_class)

metrics_gb_class_improved = classification_metrics(y_test_encoded, y_pred_gb_class_improved, y_prob_gb_class_improved)
print("Метрики классификации (улучшенный градиентный бустинг):")
for key, value in metrics_gb_class_improved.items():
    print(f"{key}: {value:.4f}")

print(f"\nКоличество деревьев: {best_gb_class.n_estimators_}")
print(f"Скорость обучения: {best_gb_class.learning_rate}")

# Улучшенная модель градиентного бустинга для регрессии
best_gb_reg = grid_gb_reg.best_estimator_
y_pred_gb_reg_improved = best_gb_reg.predict(X_test_reg)

metrics_gb_reg_improved = regression_metrics(y_test_reg, y_pred_gb_reg_improved)
print("\nМетрики регрессии (улучшенный градиентный бустинг):")
for key, value in metrics_gb_reg_improved.items():
    print(f"{key}: {value:.4f}")

print(f"\nКоличество деревьев: {best_gb_reg.n_estimators_}")
print(f"Скорость обучения: {best_gb_reg.learning_rate}")

Метрики классификации (улучшенный градиентный бустинг):
Accuracy: 0.9325
Precision: 0.9339
Recall: 0.9308
F1-Score: 0.9318
ROC-AUC: 0.9964

Количество деревьев: 50
Скорость обучения: 0.2

Метрики регрессии (улучшенный градиентный бустинг):
MSE: 81.8817
MAE: 4.3999
R2: 0.9762

Количество деревьев: 50
Скорость обучения: 0.1


## Сравнение с исходным бейзлайном

In [27]:
print("Сравнение классификации (градиентный бустинг):")
print("Метрика | Исходный | Улучшенный")
for key in metrics_gb_class.keys():
    if key in metrics_gb_class_improved:
        print(f"{key:12} | {metrics_gb_class[key]:.4f} | {metrics_gb_class_improved[key]:.4f}")

print("\nСравнение регрессии (градиентный бустинг):")
print("Метрика | Исходный | Улучшенный")
for key in metrics_gb_reg.keys():
    print(f"{key:12} | {metrics_gb_reg[key]:.4f} | {metrics_gb_reg_improved[key]:.4f}")

Сравнение классификации (градиентный бустинг):
Метрика | Исходный | Улучшенный
Accuracy     | 0.9386 | 0.9325
Precision    | 0.9405 | 0.9339
Recall       | 0.9371 | 0.9308
F1-Score     | 0.9381 | 0.9318
ROC-AUC      | 0.9969 | 0.9964

Сравнение регрессии (градиентный бустинг):
Метрика | Исходный | Улучшенный
MSE          | 127.8803 | 81.8817
MAE          | 5.8892 | 4.3999
R2           | 0.9628 | 0.9762


## Самостоятельная реализация градиентного бустинга для регрессии

### Краткая идея  
Простой базовый регрессор глубиной 1 (stump) представлялся в виде единственного разбиения по одному признаку и порогу. В листах хранились средние значения целевой переменной для соответствующих частей разбиения. Модель обучалась путём перебора всех признаков и всех уникальных порогов для каждого признака с выбором того порога, который минимизировал среднеквадратичную ошибку (MSE) на обучающей выборке.

### Формулировка задачи и критерий качества  
На каждом кандидате разбиения вычислялось предсказание $y_{pred}$ как
$$
y_{pred,i} =
\begin{cases}
\bar y_{left}, & x_i[feature\_idx] \le threshold,\\[6pt]
\bar y_{right}, & x_i[feature\_idx] > threshold,
\end{cases}
$$
где $\bar y_{left}$ и $\bar y_{right}$ — средние целевых значений в соответствующих частях. Критерий качества представлялся MSE:
$$
MSE = \frac{1}{n}\sum_{i=1}^n (y_i - y_{pred,i})^2.
$$
Выбиралось разбиение с минимальным значением $MSE$.

### Алгоритм обучения (пасcивная форма)  
1. Для каждого признака были получены уникальные значения как кандидаты порога.  
2. Для каждого порога строились маски левой и правой части; кандидаты с пустыми частями игнорировались.  
3. В каждой части вычислялось среднее целевой переменной; далее вычислялся $MSE$ для полученного stump.  
4. Сохранялось разбиение с наименьшим $MSE$; в результате в параметрах stump фиксировались $feature\_idx$, $threshold$, $left\_value$ и $right\_value$.

### Сложность и практические аспекты  
При наивной проверке всех уникальных порогов временная сложность поиска лучшего stump оценивалась как $O(n\_features \times n \times u)$, где $u$ — среднее число уникальных значений признака. Память использовалась минимально (хранились текущие средние и маски); при вычислениях применялись векторные операции для ускорения. Для данных с большим числом уникальных значений для признака число кандидатов порога могло быть большим, что увеличивало время подбора.

### Общая идея бустинга  
Аддитивная модель строилась в виде суммы $M$ простых базовых регрессоров (stump), где итоговое предсказание выражалось как
$$
\hat y^{(M)}(x) = F_0 + \eta \sum_{m=1}^M h_m(x),
$$
где $F_0$ — начальное предсказание (константа), $\eta$ — learning rate (скорость обучения), $h_m$ — m-й базовый регрессор.

### Начальное предсказание и отрицательный градиент для MSE  
Начальное предсказание выбиралось как среднее целевой переменной:
$$
F_0 = \bar y = \frac{1}{n}\sum_{i=1}^n y_i.
$$
При выборе квадратичной функции потерь $L(y,\hat y) = (y - \hat y)^2$ отрицательный градиент для примера $i$ имел вид
$$
- \frac{\partial L}{\partial \hat y_i} = y_i - \hat y_i.
$$
В реализации использовалось именно это выражение: на каждом шаге вычислялись остатки (residuals) $r_i = y_i - \hat y_i$, и на эти остатки обучался новый stump.

### Алгоритм обучения (аддитивная последовательность)  
1. Были вычислены $F_0 = \bar y$ и начальные предсказания $F^{(0)}(x_i) = F_0$ для всех $i$.  
2. Для каждой итерации $m = 1..M$ вычислялись остатки $r_i^{(m)} = y_i - F^{(m-1)}(x_i)$.  
3. На остатках обучался базовый регрессор $h_m$ (в реализации — stump), аппроксимирующий отрицательный градиент.  
4. Обновление модели выполнялось по правилу
$$
F^{(m)}(x) = F^{(m-1)}(x) + \eta \, h_m(x).
$$
5. По окончании обучения итоговое предсказание представлялось в виде суммы $F^{(M)}(x)$.

### Диагностика и числовая устойчивость  
На каждом контролируемом шаге мог вычисляться текущий $MSE = \frac{1}{n}\sum (y - F^{(m)}(x))^2$ для мониторинга сходимости. Параметр $\eta$ влиял на величину шага обновления: при больших $\eta$ наблюдалась бы быстрая адаптация, но повышался риск переобучения и нестабильности; при малых $\eta$ сходимость была бы медленнее, но итоговая модель могла оказаться более устойчивой. Для воспроизводимости результатов задавался начальный seed генератора случайных чисел.

### Сложность и ресурсы  
Обучение одного stump требовало перебора признаков и порогов (см. DecisionStumpRegressor). Для $M$ итераций сложность оценивалась как примерно $O(M \times n\_features \times n \times u)$. Память использовалась для хранения последовательности базовых моделей и текущих предсказаний.


In [28]:
import numpy as np
from collections import Counter

class DecisionStumpRegressor:
    """Простое дерево решений глубиной 1 (stump) для регрессии"""
    
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.left_value = None
        self.right_value = None
    
    def fit(self, X, y):
        """Обучение stump на данных"""
        n_samples, n_features = X.shape
        best_mse = float('inf')
        
        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold
                
                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                
                left_value = np.mean(y[left_mask])
                right_value = np.mean(y[right_mask])
                
                y_pred = np.where(left_mask, left_value, right_value)
                mse = np.mean((y - y_pred) ** 2)
                
                if mse < best_mse:
                    best_mse = mse
                    self.feature_idx = feature_idx
                    self.threshold = threshold
                    self.left_value = left_value
                    self.right_value = right_value
        
        return self
    
    def predict(self, X):
        """Предсказание значений"""
        if self.feature_idx is None:
            return np.zeros(X.shape[0])
        
        left_mask = X[:, self.feature_idx] <= self.threshold
        return np.where(left_mask, self.left_value, self.right_value)


class GradientBoostingRegressorCustom:
    """Самостоятельная реализация градиентного бустинга для регрессии"""
    
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=1, random_state=None):
        """
        Инициализация градиентного бустинга для регрессии.
        
        Параметры:
        ----------
        n_estimators : int, default=100
            Количество базовых моделей (деревьев)
        learning_rate : float, default=0.1
            Скорость обучения (шаг градиентного спуска)
        max_depth : int, default=1
            Максимальная глубина деревьев (в нашей реализации используется stump, глубина=1)
        random_state : int or None, default=None
            Случайное начальное число для воспроизводимости
        """
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.random_state = random_state
        self.models = []
        self.initial_prediction = None
        
        if random_state is not None:
            np.random.seed(random_state)
    
    def _initial_prediction(self, y):
        """Начальное предсказание (среднее значение)"""
        return np.mean(y)
    
    def _negative_gradient(self, y_true, y_pred):
        """Отрицательный градиент функции потерь (MSE)"""
        return y_true - y_pred
    
    def fit(self, X, y):
        """Обучение градиентного бустинга"""
        n_samples = X.shape[0]
        
        self.initial_prediction = self._initial_prediction(y)
        current_prediction = np.full(n_samples, self.initial_prediction)
        
        for i in range(self.n_estimators):
            residuals = self._negative_gradient(y, current_prediction)
            
            model = DecisionStumpRegressor()
            model.fit(X, residuals)
            
            self.models.append(model)
            
            current_prediction += self.learning_rate * model.predict(X)
            
            if (i + 1) % 10 == 0:
                mse = np.mean((y - current_prediction) ** 2)
                print(f"Итерация {i + 1}/{self.n_estimators}, MSE: {mse:.4f}")
        
        print(f"Обучение завершено. Обучено {self.n_estimators} моделей.")
    
    def predict(self, X):
        """Предсказание значений"""
        y_pred = np.full(X.shape[0], self.initial_prediction)
        
        for model in self.models:
            y_pred += self.learning_rate * model.predict(X)
        
        return y_pred

## Самостоятельная реализация градиентного бустинга для классификации

### Краткая идея  
Stump для классификации представлялся как одноразбиение по одному признаку и порогу с хранением наиболее частого класса в левой и правой частях. Критерием качества разбиения выступал взвешенный по размерам дочерних частей суммарный коэффициент Джини.

### Коэффициент Джини для части и критерий качества  
Для множества меток $y$ коэффициент Джини вычислялся как
$$
G(y) = 1 - \sum_k p_k^2,
$$
где $p_k$ — доля класса $k$ в подмножестве. Для кандидата порога вычислялось итоговое значение как
$$
G_{total} = \frac{n_{left}}{n} G(y_{left}) + \frac{n_{right}}{n} G(y_{right}).
$$
Выбиралось разбиение с минимальным $G_{total}$; в листах хранились наиболее частые классы для соответствующих частей.

### Алгоритм обучения (пасcивная форма)  
1. Для каждого признака перебирались уникальные значения как кандидаты порога.  
2. Для каждого порога строились маски левой и правой части; кандидаты с пустыми частями игнорировались.  
3. В каждой части вычислялся наиболее частый класс (majority class) и коэффициент Джини; далее вычислялся $G_{total}$.  
4. Сохранялось разбиение с минимальным $G_{total}$ и соответствующие классы для левой и правой ветвей.

### Сложность и практические аспекты  
Временная сложность подбора stump для классификации аналогична регрессии: $O(n\_features \times n \times u)$. Прогноз выполнялся простым применением полученного порога и подстановкой сохранённой метки соответствующей ветви.

### Подход One-vs-All и параметризация  
Многоклассовая задача была декомпозирована на $K$ бинарных задач (one-vs-all): для каждого класса $k$ строился свой набор из $M$ базовых регрессоров, аппроксимирующих логит-функцию для данного класса. Для каждого класса в качестве начального значения использовался логарифм начальных шансов:
$$
F_0^{(k)} = \log\frac{p_k}{1 - p_k + \varepsilon},
$$
где $p_k$ — доля положительных примеров для данного класса, $\varepsilon$ — малое число для числовой стабильности.

### Связь с логистической функцией и вычисление градиента  
Логит-функция и обратное отображение задавались через сигмоид:
$$
\sigma(z) = \frac{1}{1 + e^{-z}}.
$$
На каждой итерации для класса $k$ вычислялись прогнозы в логит-пространстве $F^{(m-1)}$ и преобразовывались в вероятности $\sigma(F^{(m-1)})$. В качестве отрицательного градиента использовалось
$$
r_i = y_i^{(k)} - \sigma(F^{(m-1)}(x_i)),
$$
где $y_i^{(k)}$ — бинарная метка (1 для целевого класса $k$, 0 — для остальных). На этих градиентах обучался базовый регрессор (stump), и выполнялось обновление
$$
F^{(m)}(x) = F^{(m-1)}(x) + \eta \, h_m(x).
$$

### Предсказание вероятностей и нормализация  
Для каждого класса суммировались результаты соответствующих ансамблей в логит-пространстве, затем применялась сигмоида для получения не нормализованных оценок вероятности по классам:
$$
\tilde p_k(x) = \sigma\big(F^{(M)}_k(x)\big).
$$
Далее выполнялась пост-нормализация по классам:
$$
p_k(x) = \frac{\tilde p_k(x)}{\sum_j \tilde p_j(x)}.
$$
В реализации сигмоидные выходы для каждого класса сначала вычислялись и затем нормировались по сумме. Отмечалось, что такая пост-нормализация сигмоидных скорингов не является строгим эквивалентом многоклассового softmax, но обеспечивает нормированные вероятности, пригодные для выбора класса по максимальной вероятности.

### Алгоритм обучения (по классам)  
1. Для каждого класса $k$ была сформирована бинарная целевая переменная $y^{(k)}$.  
2. Начальное логит-предсказание $F_0^{(k)}$ вычислялось по частоте позитивных примеров.  
3. Для $m=1..M$ вычислялся текущий отрицательный градиент $y^{(k)} - \sigma(F^{(m-1)}(x))$, на котором обучался stump.  
4. Обновление логит-функции выполнялось добавлением $\eta \, h_m(x)$.  
5. Для контроля верифицировалась точность бинарной постановки на валидационной части или по порогу $0.5$ в сигмоиде.

### Числовые меры устойчивости  
При вычислении сигмоиды использовалось ограничение аргумента (клиппинг), а при вычислении логитов — добавление малого $\varepsilon$ в знаменатель для предотвращения деления на ноль: это обеспечивало устойчивость для очень малых или очень больших долей и исключало `inf`/`nan`.

### Сложность и ресурсы  
Обучение каждого бинарного ансамбля требовало $M$ обучений stump'ов; для всех классов общее число базовых моделей оценивалось как $K \times M$. Общая вычислительная нагрузка масштабировалась линейно по числу классов, количеству итераций и сложности обучения одного stump'а.


In [29]:
class DecisionStumpClassifier:
    """Простое дерево решений глубиной 1 (stump) для классификации"""
    
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.left_class = None
        self.right_class = None
    
    def fit(self, X, y):
        """Обучение stump на данных"""
        n_samples, n_features = X.shape
        best_gini = float('inf')
        
        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold
                
                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                
                left_classes = y[left_mask]
                right_classes = y[right_mask]
                
                left_class = Counter(left_classes).most_common(1)[0][0]
                right_class = Counter(right_classes).most_common(1)[0][0]
                
                # Вычисление коэффициента Джини
                left_gini = self._gini(left_classes)
                right_gini = self._gini(right_classes)
                
                n_left = len(left_classes)
                n_right = len(right_classes)
                total_gini = (n_left / n_samples) * left_gini + (n_right / n_samples) * right_gini
                
                if total_gini < best_gini:
                    best_gini = total_gini
                    self.feature_idx = feature_idx
                    self.threshold = threshold
                    self.left_class = left_class
                    self.right_class = right_class
        
        return self
    
    def _gini(self, y):
        """Вычисление коэффициента Джини"""
        if len(y) == 0:
            return 0
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)
    
    def predict(self, X):
        """Предсказание классов"""
        if self.feature_idx is None:
            return np.zeros(X.shape[0], dtype=int)
        
        left_mask = X[:, self.feature_idx] <= self.threshold
        predictions = np.where(left_mask, self.left_class, self.right_class)
        return predictions


class GradientBoostingClassifierCustom:
    """Самостоятельная реализация градиентного бустинга для классификации (One-vs-All)"""
    
    def __init__(self, n_estimators=100, learning_rate=0.1, random_state=None):
        """
        Инициализация градиентного бустинга для классификации.
        
        Параметры:
        ----------
        n_estimators : int, default=100
            Количество базовых моделей (деревьев)
        learning_rate : float, default=0.1
            Скорость обучения (шаг градиентного спуска)
        random_state : int or None, default=None
            Случайное начальное число для воспроизводимости
        """
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.models = []
        self.classes_ = None
        self.n_classes_ = None
        
        if random_state is not None:
            np.random.seed(random_state)
    
    def _sigmoid(self, x):
        """Сигмоидная функция"""
        return 1 / (1 + np.exp(-np.clip(x, -250, 250)))
    
    def fit(self, X, y):
        """Обучение градиентного бустинга (One-vs-All подход)"""
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        n_samples = X.shape[0]
        
        self.models = [[] for _ in range(self.n_classes_)]
        
        for class_idx, class_label in enumerate(self.classes_):
            print(f"Обучение модели для класса {class_label}...")
            
            # Преобразование задачи в бинарную (One-vs-All)
            y_binary = (y == class_label).astype(float)
            
            prob_positive = np.mean(y_binary)
            initial_prediction = np.log(prob_positive / (1 - prob_positive + 1e-10))
            current_prediction = np.full(n_samples, initial_prediction)
            
            for i in range(self.n_estimators):
                probabilities = self._sigmoid(current_prediction)
                
                gradients = y_binary - probabilities
                
                model = DecisionStumpRegressor()
                model.fit(X, gradients)
                
                self.models[class_idx].append(model)
                
                current_prediction += self.learning_rate * model.predict(X)
                
                if (i + 1) % 20 == 0:
                    predictions = (self._sigmoid(current_prediction) > 0.5).astype(int)
                    accuracy = np.mean(predictions == y_binary)
                    print(f"  Класс {class_label}, итерация {i + 1}/{self.n_estimators}, Accuracy: {accuracy:.4f}")
        
        print(f"Обучение завершено. Обучено {self.n_estimators} моделей для каждого из {self.n_classes_} классов.")
    
    def predict_proba(self, X):
        """Предсказание вероятностей для каждого класса"""
        n_samples = X.shape[0]
        proba = np.zeros((n_samples, self.n_classes_))
        
        for class_idx in range(self.n_classes_):
            current_prediction = np.zeros(n_samples)
            
            for model in self.models[class_idx]:
                current_prediction += self.learning_rate * model.predict(X)
            
            proba[:, class_idx] = self._sigmoid(current_prediction)
        
        proba_sum = proba.sum(axis=1, keepdims=True)
        proba = proba / proba_sum
        
        return proba
    
    def predict(self, X):
        """Предсказание классов"""
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

### Обучение и оценка полностью самостоятельной реализации градиентного бустинга для регрессии

Используем оптимальные гиперпараметры, найденные при улучшении бейзлайна.

In [30]:
# Получаем лучшие параметры из GridSearchCV
best_params_gb_reg = grid_gb_reg.best_params_

print("Обучение полностью самостоятельной реализации градиентного бустинга для регрессии...")

# Адаптация параметров под нашу реализацию
n_estimators_custom = min(50, best_params_gb_reg.get('n_estimators', 100))  # Уменьшаем для ускорения
learning_rate_custom = best_params_gb_reg.get('learning_rate', 0.1)

gb_custom_reg = GradientBoostingRegressorCustom(
    n_estimators=n_estimators_custom,
    learning_rate=learning_rate_custom,
    max_depth=1,  # В нашей реализации используются stumps (глубина=1)
    random_state=42
)

gb_custom_reg.fit(X_train_reg.values, y_train_reg.values)

# Прогнозы
y_pred_gb_custom_reg = gb_custom_reg.predict(X_test_reg.values)

# Оценка метрик
metrics_gb_custom_reg = regression_metrics(y_test_reg, y_pred_gb_custom_reg)
print("\nМетрики полностью самостоятельной реализации градиентного бустинга для регрессии:")
for key, value in metrics_gb_custom_reg.items():
    print(f"{key}: {value:.4f}")

Обучение полностью самостоятельной реализации градиентного бустинга для регрессии...
Итерация 10/50, MSE: 1293.8331
Итерация 20/50, MSE: 717.8474
Итерация 30/50, MSE: 498.5078
Итерация 40/50, MSE: 388.8940
Итерация 50/50, MSE: 328.6082
Обучение завершено. Обучено 50 моделей.

Метрики полностью самостоятельной реализации градиентного бустинга для регрессии:
MSE: 369.4764
MAE: 11.1986
R2: 0.8926


### Обучение и оценка полностью самостоятельной реализации градиентного бустинга для классификации

In [31]:
# Получаем лучшие параметры из GridSearchCV
best_params_gb_class = grid_gb_class.best_params_

print("Обучение полностью самостоятельной реализации градиентного бустинга для классификации...")

# Адаптация параметров под нашу реализацию
n_estimators_custom_class = min(30, best_params_gb_class.get('n_estimators', 100))  # Уменьшаем для ускорения
learning_rate_custom_class = best_params_gb_class.get('learning_rate', 0.1)

gb_custom_class = GradientBoostingClassifierCustom(
    n_estimators=n_estimators_custom_class,
    learning_rate=learning_rate_custom_class,
    random_state=42
)

gb_custom_class.fit(X_train_class.values, y_train_encoded)

# Прогнозы
y_pred_gb_custom = gb_custom_class.predict(X_test_class.values)
y_prob_gb_custom = gb_custom_class.predict_proba(X_test_class.values)

# Оценка метрик
metrics_gb_custom_class = classification_metrics(y_test_encoded, y_pred_gb_custom, y_prob_gb_custom)
print("\nМетрики полностью самостоятельной реализации градиентного бустинга для классификации:")
for key, value in metrics_gb_custom_class.items():
    print(f"{key}: {value:.4f}")

Обучение полностью самостоятельной реализации градиентного бустинга для классификации...
Обучение модели для класса 0...
  Класс 0, итерация 20/30, Accuracy: 1.0000
Обучение модели для класса 1...
  Класс 1, итерация 20/30, Accuracy: 0.8251
Обучение модели для класса 2...
  Класс 2, итерация 20/30, Accuracy: 0.8131
Обучение модели для класса 3...
  Класс 3, итерация 20/30, Accuracy: 0.8332
Обучение модели для класса 4...
  Класс 4, итерация 20/30, Accuracy: 0.9619
Обучение модели для класса 5...
  Класс 5, итерация 20/30, Accuracy: 0.8541
Обучение завершено. Обучено 30 моделей для каждого из 6 классов.

Метрики полностью самостоятельной реализации градиентного бустинга для классификации:
Accuracy: 0.8381
Precision: 0.8503
Recall: 0.8322
F1-Score: 0.8334
ROC-AUC: 0.9737


## Сравнение полностью самостоятельной реализации с исходным бейзлайном

In [32]:
print("Сравнение полностью самостоятельной реализации градиентного бустинга для классификации с исходным бейзлайном:")
print("Метрика | Самостоятельная | Исходный бейзлайн")
for key in metrics_gb_custom_class.keys():
    if key in metrics_gb_class:
        print(f"{key:12} | {metrics_gb_custom_class[key]:.4f} | {metrics_gb_class[key]:.4f}")

print("\nСравнение полностью самостоятельной реализации градиентного бустинга для регрессии с исходным бейзлайном:")
print("Метрика | Самостоятельная | Исходный бейзлайн")
for key in metrics_gb_custom_reg.keys():
    print(f"{key:12} | {metrics_gb_custom_reg[key]:.4f} | {metrics_gb_reg[key]:.4f}")

Сравнение полностью самостоятельной реализации градиентного бустинга для классификации с исходным бейзлайном:
Метрика | Самостоятельная | Исходный бейзлайн
Accuracy     | 0.8381 | 0.9386
Precision    | 0.8503 | 0.9405
Recall       | 0.8322 | 0.9371
F1-Score     | 0.8334 | 0.9381
ROC-AUC      | 0.9737 | 0.9969

Сравнение полностью самостоятельной реализации градиентного бустинга для регрессии с исходным бейзлайном:
Метрика | Самостоятельная | Исходный бейзлайн
MSE          | 369.4764 | 127.8803
MAE          | 11.1986 | 5.8892
R2           | 0.8926 | 0.9628


## Сравнение полностью самостоятельной реализации с улучшенным бейзлайном

In [33]:
print("Сравнение полностью самостоятельной реализации градиентного бустинга для классификации с улучшенным бейзлайном:")
print("Метрика | Самостоятельная | Улучшенный бейзлайн")
for key in metrics_gb_custom_class.keys():
    if key in metrics_gb_class_improved:
        print(f"{key:12} | {metrics_gb_custom_class[key]:.4f} | {metrics_gb_class_improved[key]:.4f}")

print("\nСравнение полностью самостоятельной реализации градиентного бустинга для регрессии с улучшенным бейзлайном:")
print("Метрика | Самостоятельная | Улучшенный бейзлайн")
for key in metrics_gb_custom_reg.keys():
    print(f"{key:12} | {metrics_gb_custom_reg[key]:.4f} | {metrics_gb_reg_improved[key]:.4f}")

Сравнение полностью самостоятельной реализации градиентного бустинга для классификации с улучшенным бейзлайном:
Метрика | Самостоятельная | Улучшенный бейзлайн
Accuracy     | 0.8381 | 0.9325
Precision    | 0.8503 | 0.9339
Recall       | 0.8322 | 0.9308
F1-Score     | 0.8334 | 0.9318
ROC-AUC      | 0.9737 | 0.9964

Сравнение полностью самостоятельной реализации градиентного бустинга для регрессии с улучшенным бейзлайном:
Метрика | Самостоятельная | Улучшенный бейзлайн
MSE          | 369.4764 | 81.8817
MAE          | 11.1986 | 4.3999
R2           | 0.8926 | 0.9762


## Итоговые выводы по лабораторной работе

### 1. Выбор данных и метрик

**Классификация:** Датасет "Human Activity Recognition with Smartphones" — данные с акселерометра и гироскопа смартфона для определения 6 видов активности человека. Практическая задача: мониторинг физической активности в приложениях здоровья и фитнеса.

**Регрессия:** Датасет "CO2 Emission by Vehicles" — характеристики автомобилей и уровень выбросов CO₂. Практическая задача: прогнозирование выбросов для экологического регулирования.

**Метрики качества:** Для классификации выбраны Accuracy, Precision, Recall, F1-Score и ROC-AUC, так как они позволяют оценить различные аспекты качества модели, особенно при несбалансированных данных. Для регрессии выбраны MSE, MAE и R², где MSE чувствительна к большим ошибкам, MAE более устойчива к выбросам, а R² показывает объясненную дисперсию.

### 2. Бейзлайн и его улучшение

**Исходный бейзлайн показал следующие результаты:**

**Градиентный бустинг (классификация):** Accuracy = 0.9386, ROC-AUC = 0.9969, количество деревьев = 100

**Градиентный бустинг (регрессия):** R² = 0.9628, MSE = 127.8803, количество деревьев = 100

**Для градиентного бустинга масштабирование признаков не требуется**, так как алгоритм использует деревья решений.

**Подбор гиперпараметров на кросс-валидации дал следующие результаты:**

**Градиентный бустинг (классификация):** Лучшие параметры: {'learning_rate': 0.2, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 50, 'subsample': 0.8}, Accuracy на кросс-валидации составила 0.8995, что на 4.17% ниже исходного бейзлайна.

**Градиентный бустинг (регрессия):** Лучшие параметры: {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 50, 'subsample': 1.0}, MSE на кросс-валидации составила 71.7056, что на 43.9% лучше исходного бейзлайна.

**Улучшенные модели на тестовой выборке показали следующие результаты:**

**Градиентный бустинг (классификация):** Accuracy = 0.9325 (на 0.65% ниже относительно исходного)

**Градиентный бустинг (регрессия):** R² = 0.9762 (на 1.39% лучше относительно исходного), MSE = 81.8817 (на 35.96% лучше относительно исходного)

**Особенность градиентного бустинга:** В отличие от случайного леса, который строит деревья независимо, градиентный бустинг строит деревья последовательно, каждое следующее дерево учится на ошибках предыдущих. Это позволяет достичь очень высокой точности, но требует более тщательной настройки гиперпараметров.

### 3. Самостоятельная реализация алгоритмов

Реализованы алгоритмы градиентного бустинга для классификации и регрессии "с нуля" с использованием только базового Python и NumPy.

**Включены ключевые особенности градиентного бустинга:**
1. Последовательное обучение моделей (бустинг)
2. Использование градиентов функции потерь для обучения следующих моделей
3. Для регрессии: использование остатков (negative gradients) как целевых переменных
4. Для классификации: One-vs-All подход с использованием логистической функции потерь
5. Использование простых деревьев глубиной 1 (stumps) в качестве базовых моделей

**Результаты самостоятельной реализации:**

**Градиентный бустинг (классификация):** Accuracy = 0.8381 (сравнение с sklearn: хуже на 10.71%)

**Градиентный бустинг (регрессия):** R² = 0.8926 (сравнение с sklearn: хуже на 7.29%), MSE = 369.4764 (сравнение: хуже на 188.97%)

**Особенности реализации:** Для ускорения обучения количество деревьев в самостоятельной реализации было уменьшено (30 для классификации, 50 для регрессии) по сравнению с sklearn. Также использовались упрощенные деревья глубиной 1 (stumps) вместо полноценных деревьев. Несмотря на это, модель показала удовлетворительные результаты, что подтверждает понимание принципов работы алгоритма, но также демонстрирует важность оптимизации и использования более сложных базовых моделей.

### 4. Сравнение всех подходов

**Классификация (градиентный бустинг):**

Исходный бейзлайн: Accuracy = 0.9386

Улучшенный бейзлайн: Accuracy = 0.9325 (на 0.65% ниже относительно исходного)

Самостоятельная реализация: Accuracy = 0.8381 (на 10.71% ниже относительно исходного, на 10.13% ниже относительно улучшенного)

**Регрессия (градиентный бустинг):**

Исходный бейзлайн: R² = 0.9628, MSE = 127.8803

Улучшенный бейзлайн: R² = 0.9762 (на 1.39% лучше), MSE = 81.8817 (на 35.96% лучше)

Самостоятельная реализация: R² = 0.8926 (на 7.29% ниже относительно исходного), MSE = 369.4764 (на 188.97% выше)

### 5. Ключевые наблюдения и выводы

**Высокая точность градиентного бустинга:** Градиентный бустинг продемонстрировал отличные результаты на обеих задачах, особенно для регрессии, где улучшенная модель достигла R² = 0.9762. Это связано с его способностью последовательно исправлять ошибки предыдущих моделей.

**Важность скорости обучения:** Параметр learning_rate критически важен для градиентного бустинга. В нашем случае оптимальные значения составили 0.2 для классификации и 0.1 для регрессии.

**Устойчивость к переобучению:** Градиентный бустинг может быть склонен к переобучению, особенно при большом количестве деревьев. В нашей работе использование subsample = 0.8 для классификации помогло снизить риск переобучения.

**Самостоятельная реализация:** Показала удовлетворительные результаты, особенно учитывая упрощения (деревья глубиной 1 и меньшее количество итераций). Accuracy = 0.8381 для классификации и R² = 0.8926 для регрессии подтверждают понимание основных принципов градиентного бустинга, но также показывают важность оптимизированных реализаций.

**Вычислительная сложность:** Градиентный бустинг требует значительных вычислительных ресурсов, особенно для большого количества деревьев. Наша реализация на чистом Python оказалась значительно медленнее оптимизированной библиотечной реализации.

Работа продемонстрировала как теоретические основы градиентного бустинга, так и практические аспекты его применения, подтвердив эффективность последовательных ансамблевых методов для достижения высокой точности предсказаний.